# cuda-jupyter

Run a cell (**Shift+Enter**) and the CUDA it corresponds to appears on the right:

- **PyTorch, JAX, TensorFlow**: every op is traced and translated into CUDA C++ kernels and host launches for the exact shapes that ran.
- **Numba, Triton, CuPy**: you write (or generate) real GPU kernels, so you see the PTX / CUDA C++ they compiled to.

Hover a CUDA line to see which Python line produced it; move the cursor in a cell to do the reverse. The terminal below is a real shell on the same machine.

The framework sections at the end need a CUDA GPU (`./run_t4` runs everything on a Colab T4).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, device)

## Elementwise ops
Each op becomes one elementwise kernel, one thread per element. Python scalars become kernel parameters.

In [ ]:
x = torch.randn(1024, 1024, device=device)
y = torch.relu(x) * 2 + 1
y.mean()

## Broadcasting
Shapes that do not match become *strided* kernels: broadcast dims get stride 0.

In [ ]:
bias = torch.randn(1024, device=device)
z = torch.sigmoid(x + bias)
z.shape

## Matmul + softmax
`@` lowers to a tiled shared-memory GEMM (with the equivalent cuBLAS call alongside); softmax is a one-block-per-row kernel with warp-shuffle reductions.

In [ ]:
a = torch.randn(256, 512, device=device)
b = torch.randn(512, 128, device=device)
c = a @ b
p = F.softmax(c, dim=-1)
p.sum(dim=-1)[:4]

## A small MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

model = MLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
xb = torch.randn(64, 784, device=device)
yb = torch.randint(0, 10, (64,), device=device)
model

## Training steps
Forward, backward (autograd's kernels are labelled with their backward node) and the SGD update. Loop iterations that repeat are collapsed and reuse the same buffers.

In [ ]:
for step in range(3):
    loss = F.cross_entropy(model(xb), yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"loss = {loss.item():.4f}")

## Conv → GELU → pool → LayerNorm

In [ ]:
conv = nn.Conv2d(3, 16, 3, padding=1).to(device)
ln = nn.LayerNorm(16).to(device)
img = torch.randn(8, 3, 32, 32, device=device)
h = F.max_pool2d(F.gelu(conv(img)), 2)
out = ln(h.permute(0, 2, 3, 1))
out.shape

## JAX
`jax.jit` functions are traced op by op (with `jax.disable_jit`) so you can see every kernel; XLA would fuse many of them.

In [ ]:
import jax
import jax.numpy as jnp

W = jax.random.normal(jax.random.key(0), (256, 128))
xj = jnp.ones((64, 256))

@jax.jit
def mlp(x, W):
    return jax.nn.gelu(x @ W).sum(axis=-1)

loss = lambda W: jnp.mean(mlp(xj, W) ** 2)
gW = jax.grad(loss)(W)
gW.shape

## TensorFlow / Keras
Eager ops and `tf.function`s (run eagerly while tracing), including the gradient tape's backward ops.

In [ ]:
import tensorflow as tf

xt = tf.random.normal((64, 256))
dense = tf.keras.layers.Dense(128, activation="relu")
with tf.GradientTape() as tape:
    out_tf = tf.reduce_mean(dense(xt) ** 2)
grads = tape.gradient(out_tf, dense.trainable_variables)
[g.shape for g in grads]

## Numba CUDA
A hand-written kernel: the pane shows the PTX that NVVM generated, mapped back to the Python lines.

In [ ]:
from numba import cuda
import numpy as np

@cuda.jit
def saxpy(a, x, y, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = a * x[i] + y[i]

n = 1 << 20
xn = cuda.to_device(np.random.rand(n).astype(np.float32))
yn = cuda.to_device(np.random.rand(n).astype(np.float32))
outn = cuda.device_array_like(xn)
saxpy[(n + 255) // 256, 256](2.0, xn, yn, outn)
outn.copy_to_host()[:4]

## Triton
A fused row softmax in Triton: its PTX appears next to the CUDA for the surrounding PyTorch ops.

In [ ]:
import triton
import triton.language as tl

@triton.jit
def softmax_kernel(X, Y, n_cols, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols
    x = tl.load(X + row * n_cols + cols, mask=mask, other=-float("inf"))
    x = x - tl.max(x, axis=0)
    e = tl.exp(x)
    tl.store(Y + row * n_cols + cols, e / tl.sum(e, axis=0), mask=mask)

xs = torch.randn(128, 1000, device="cuda")
ys = torch.empty_like(xs)
softmax_kernel[(xs.shape[0],)](xs, ys, xs.shape[1], BLOCK=1024)
torch.allclose(ys, torch.softmax(xs, -1), atol=1e-6)

## CuPy
CuPy generates CUDA C++ for its ufuncs, reductions and custom kernels and compiles it with NVRTC; that source is shown as-is.

In [ ]:
import cupy as cp

ca = cp.random.rand(1_000_000, dtype=cp.float32)
squared_diff = cp.ElementwiseKernel(
    "float32 x, float32 m", "float32 y", "y = (x - m) * (x - m)", "squared_diff")
variance = squared_diff(ca, ca.mean()).mean()
float(variance)